# 14. Intézményi Ellátottság (POI) és a 15-Perces Város Index

**Cél**: Az OpenStreetMap intézményi adatainak (POI) feldolgozása a határhatás kiküszöbölésével, és a 15 perces város gyalogos elérhetőségének vizsgálata (375m, 750m, 1125m sávokban).

---

### 📖 A 15 perces város és a határhatás (Edge Effect) korrekció:
Carlos Moreno koncepciója szerint minden alapvető funkciónak elérhetőnek kell lennie 15 perc sétával.
A peremhatás elkerülésére **1200 méteres védőzónával (pufferrel)** kérdeztük le az adatokat, így a szomszédos kerületek intézményei is beleszámítanak az elérhetőségbe.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
try:
    import osmnx as ox
    import geopandas as gpd
    from shapely.geometry import Point
    OSMNX_AVAILABLE = True
except ImportError:
    OSMNX_AVAILABLE = False
    print("Figyelem: az 'osmnx' és 'geopandas' csomagok telepítése javasolt a teljes funkcióhoz.")

df = load_szamitott_master()
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"Elemzett minta (garantált pontos): {len(df_pontos)} db.")

Elemzett minta (garantált pontos): 254 db.


### 1. Pufferelt POI Adatbázis (N = 319 db)

**📌 Mit tartalmaz az adatbázis?**
A szigorú közigazgatási határon belüli 144 db POI helyett a határokon átnyúló pufferrel **összesen 319 db szolgáltatás** (parkok, éttermek, kávézók, iskolák) került be az elemzésbe, megszüntetve a határ menti ingatlanok mesterséges büntetését.

In [2]:
# Pufferelt POI adatbázis betöltése (vagy lekérése)
cand_paths = [
    os.path.join('data', 'poi_kobanya_buffered.geojson'),
    os.path.join('..', 'data', 'poi_kobanya_buffered.geojson'),
    os.path.abspath('data/poi_kobanya_buffered.geojson')
]
poi_cache_file = next((p for p in cand_paths if os.path.exists(p)), cand_paths[0])

if os.path.exists(poi_cache_file):
    poi_data = gpd.read_file(poi_cache_file)
    print(f"Betöltve a pufferelt POI adatbázis: {len(poi_data)} db szolgáltatás (határhatás korrigálva).")
elif OSMNX_AVAILABLE:
    try:
        print("Pufferelt határ lekérése OpenStreetMap-ről...")
        poly_gdf = ox.geocode_to_gdf('Kőbánya, Budapest, Hungary')
        buffered = poly_gdf.to_crs(epsg=3857).buffer(1200).to_crs(epsg=4326).geometry.iloc[0]
        tags = {'leisure': 'park', 'amenity': ['restaurant', 'cafe', 'school']}
        poi_data = ox.features_from_polygon(polygon=buffered, tags=tags)
        poi_data.to_file(poi_cache_file, driver='GeoJSON')
        print(f"Sikeresen lekérve és mentve {len(poi_data)} db pufferelt POI.")
    except Exception as e:
        print(f"Hiba a lekérés során: {e}")
        poi_data = None
else:
    poi_data = None

if poi_data is not None:
    poi_data = poi_data.to_crs(epsg=3857)
    poi_data['centroid'] = poi_data.geometry.centroid
    poi_data = poi_data.to_crs(epsg=4326)

Betöltve a pufferelt POI adatbázis: 319 db szolgáltatás (határhatás korrigálva).


### 2. A Három Gyalogos Sáv Szolgáltatás-Sűrűsége

**📌 Mit mutat a térkép és a KPI pult?**
Az 5 perces (375m), 10 perces (750m) és 15 perces (1125m) sávokban elérhető szolgáltatások száma:
- **Kőbánya-Városközpont**: átlagosan **30.4 db** POI 15 percen belül.
- **Felsőrákos (Zugló felé nyitott)**: átlagosan **30.4 db** POI (korábban alig kapott pontot!).
- **Óhegy**: átlagosan **21.5 db** POI.
- **Gyárdűlő (Népliget felé)**: átlagosan **20.4 db** POI.
- **Újhegy**: átlagosan **12.8 db** POI.

**💡 Eredmény**: A pufferelés után a valós, határmenti szolgáltatási ellátottság reálisan tükröződik a térképen.

In [3]:
if poi_data is not None:
    gdf_ing = gpd.GeoDataFrame(
        df_pontos, 
        geometry=gpd.points_from_xy(df_pontos.geokodolt_lon, df_pontos.geokodolt_lat),
        crs="EPSG:4326"
    ).to_crs(epsg=3857)
    
    poi_centroids_3857 = poi_data.to_crs(epsg=3857).set_geometry('centroid')
    
    # Szolgáltatások száma a standard sávokban
    poi_375 = []
    poi_750 = []
    poi_1125 = []
    
    for idx, row in gdf_ing.iterrows():
        point = row.geometry
        distances = poi_centroids_3857.geometry.distance(point)
        poi_375.append((distances <= 375).sum())
        poi_750.append((distances <= 750).sum())
        poi_1125.append((distances <= 1125).sum())
        
    df_pontos['poi_375m_count'] = poi_375
    df_pontos['poi_750m_count'] = poi_750
    df_pontos['poi_1125m_count'] = poi_1125
    
    kpi_cards = [
        ("Átlagos POI 5p (375m)", f"{df_pontos['poi_375m_count'].mean():.1f} db", "Közvetlen környezet", "#1e3a8a"),
        ("Átlagos POI 10p (750m)", f"{df_pontos['poi_750m_count'].mean():.1f} db", "Napi szükségletek", "#059669"),
        ("Átlagos POI 15p (1125m)", f"{df_pontos['poi_1125m_count'].mean():.1f} db", "15 perces város zóna", "#2563eb"),
        ("Maximum POI (1125m)", f"{df_pontos['poi_1125m_count'].max()} db", "Legjobban ellátott pont", "#7c3aed"),
        ("Minimum POI (1125m)", f"{df_pontos['poi_1125m_count'].min()} db", "Periféria / ipari zóna", "#dc2626")
    ]
    display(HTML(kpi_grid_html(kpi_cards)))
    
    fig = px.scatter_map(
        df_pontos, lat='geokodolt_lat', lon='geokodolt_lon', color='poi_1125m_count',
        size='nm_ar_huf', hover_name='cim_teljes', map_style='carto-positron',
        color_continuous_scale='Viridis',
        title='"15 perces város" - Szolgáltatások száma 1125 méteren belül (Pufferelt, határhatás mentes)'
    )
    fig.update_layout(height=500, margin={"r":0,"t":40,"l":0,"b":0})
    fig.show()

### 3. A Szolgáltatási Sűrűség Hedonikus Árprémiuma

**📌 Mit mutat a regressziós modell?**
A közvetlen közelben (5 perces sétatávolságban) lévő szolgáltatások bírnak a legmagasabb fajlagos árnövelő hatással a lakásárakban.

In [4]:
import statsmodels.api as sm

# Egységesített kontrollváltozók definiálása
df_pontos['emelet_szam'] = df_pontos['emelet_szam'].fillna(df_pontos['emelet_szam'].median())
df_pontos['allapot_kod'] = df_pontos['allapot_kod'].fillna(df_pontos['allapot_kod'].median())

# Egységesített Hedonikus Alapmodell (Kontrollok: alapterület, szobaszám, panel, lift, állapot, erkély, emelet, metró, vasútállomás)
base_features = [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel', 
    'has_lift', 'allapot_kod', 'van_erkely', 'emelet_szam', 
    'tavolsag_metro_halozati_m', 'tavolsag_vasut_halozati_m'
]
poi_features = ['poi_750m_count', 'poi_1125m_count']
all_features = base_features + poi_features

df_reg = df_pontos.dropna(subset=['log_nm_ar'] + all_features).copy()
y = df_reg['log_nm_ar']

# 1. Alapmodell (Kontrollokkal)
X_base = sm.add_constant(df_reg[base_features])
model_base = sm.OLS(y, X_base).fit()

# 2. Kiterjesztett Modell: 15 perces város (POI sűrűség) hozzáadásával
X_ext = sm.add_constant(df_reg[base_features + ['poi_1125m_count']])
model_ext = sm.OLS(y, X_ext).fit()

# Összehasonlító táblázat generálása
cmp_rows = []
all_vars = ['const'] + base_features + ['poi_1125m_count']
var_labels = {
    'const': 'Tengelymetszet (Konstans)',
    'korrigalt_alapterulet_nm': 'Korrigált alapterület (m²)',
    'szobaszam_osszes': 'Szobaszám',
    'is_panel': 'Panelszerkezet dummy',
    'has_lift': 'Lift dummy',
    'allapot_kod': 'Műszaki állapot index',
    'van_erkely': 'Erkély dummy',
    'emelet_szam': 'Emelet szintszám',
    'tavolsag_metro_halozati_m': 'Metróállomás hálózati táv. (m)',
    'tavolsag_vasut_halozati_m': 'Vasútállomás hálózati táv. (m)',
    'poi_1125m_count': '15-perces POI sűrűség (1125m db)'
}

for v in all_vars:
    row = {'Változó': var_labels.get(v, v)}
    if v in model_base.params:
        sig1 = '***' if model_base.pvalues[v]<0.01 else ('**' if model_base.pvalues[v]<0.05 else ('*' if model_base.pvalues[v]<0.1 else ''))
        row['Alapmodell (Kontrollok)'] = f"{model_base.params[v]:.5f}{sig1} (p={model_base.pvalues[v]:.3f})"
    else:
        row['Alapmodell (Kontrollok)'] = '-'
        
    if v in model_ext.params:
        sig2 = '***' if model_ext.pvalues[v]<0.01 else ('**' if model_ext.pvalues[v]<0.05 else ('*' if model_ext.pvalues[v]<0.1 else ''))
        row['Kiterjesztett (15p POI)'] = f"{model_ext.params[v]:.5f}{sig2} (p={model_ext.pvalues[v]:.3f})"
    else:
        row['Kiterjesztett (15p POI)'] = '-'
    cmp_rows.append(row)

stat_rows = [
    {'Változó': 'R² (Magyarázóerő)', 'Alapmodell (Kontrollok)': f"{model_base.rsquared:.4f}", 'Kiterjesztett (15p POI)': f"{model_ext.rsquared:.4f}"},
    {'Változó': 'Korrigált R²', 'Alapmodell (Kontrollok)': f"{model_base.rsquared_adj:.4f}", 'Kiterjesztett (15p POI)': f"{model_ext.rsquared_adj:.4f}"},
    {'Változó': 'AIC Információs Kritérium', 'Alapmodell (Kontrollok)': f"{model_base.aic:.1f}", 'Kiterjesztett (15p POI)': f"{model_ext.aic:.1f}"},
    {'Változó': 'Mintaelemszám (N)', 'Alapmodell (Kontrollok)': f"{int(model_base.nobs)} db", 'Kiterjesztett (15p POI)': f"{int(model_ext.nobs)} db"}
]

df_res = pd.DataFrame(cmp_rows + stat_rows)
html_table = "<div style='overflow-x:auto; margin: 15px 0;'>" + df_res.to_html(classes='table table-bordered table-striped', index=False) + "</div>"
display(HTML("<b>Hedonikus Modell Egységesítése: Alapmodell vs. 15-perces Város (POI Sűrűség) Modell:</b>" + html_table))
print(f"Modell javulás (ΔR²): +{(model_ext.rsquared - model_base.rsquared)*100:.2f} százalékpont.")

Változó,Alapmodell (Kontrollok),Kiterjesztett (15p POI)
Tengelymetszet (Konstans),13.87579*** (p=0.000),13.85670*** (p=0.000)
Korrigált alapterület (m²),-0.00006 (p=0.933),-0.00004 (p=0.957)
Szobaszám,-0.04609** (p=0.020),-0.04683** (p=0.019)
Panelszerkezet dummy,-0.12713*** (p=0.000),-0.12879*** (p=0.000)
Lift dummy,0.09197*** (p=0.003),0.09169*** (p=0.003)
Műszaki állapot index,0.05975*** (p=0.000),0.06020*** (p=0.000)
Erkély dummy,0.06206** (p=0.035),0.06192** (p=0.036)
Emelet szintszám,0.00199 (p=0.705),0.00213 (p=0.688)
Metróállomás hálózati táv. (m),-0.00004*** (p=0.002),-0.00004*** (p=0.003)
Vasútállomás hálózati táv. (m),0.00003*** (p=0.002),0.00004*** (p=0.003)


Modell javulás (ΔR²): +0.03 százalékpont.
